In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import json

df = pd.read_csv("../data/processed/train_scaled.csv")

with open("../data/processed/top_sensors.json") as f:
    sensors = json.load(f)

In [2]:
from sklearn.model_selection import train_test_split

engine_ids = df["engine_id"].unique()

train_engines, test_engines = train_test_split(
    engine_ids, test_size=0.2, random_state=42
)

train_df = df[df["engine_id"].isin(train_engines)]
test_df = df[df["engine_id"].isin(test_engines)]

In [3]:
def create_sequences(df, sensors, window):

    X = []
    y = []

    for engine in df["engine_id"].unique():

        engine_df = df[df["engine_id"] == engine].sort_values("cycle")

        sensor_vals = engine_df[sensors].values
        rul_vals = engine_df["RUL"].values

        for i in range(len(engine_df) - window + 1):

            seq = sensor_vals[i:i+window]   # NO FLATTEN
            target = rul_vals[i+window-1]

            X.append(seq)
            y.append(target)

    return np.array(X), np.array(y)

In [4]:
window = 60

X_train, y_train = create_sequences(train_df, sensors, window)
X_test, y_test = create_sequences(test_df, sensors, window)

np.savez(
    "../data/processed/lstm_window_60.npz",
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test
)

print(X_train.shape)
print(X_test.shape)

(11841, 60, 10)
(2890, 60, 10)


In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.metrics import mean_squared_error

In [6]:
data = np.load("../data/processed/lstm_window_60.npz")

X_train = torch.tensor(data["X_train"], dtype=torch.float32)
y_train = torch.tensor(data["y_train"], dtype=torch.float32).view(-1,1)

X_test = torch.tensor(data["X_test"], dtype=torch.float32)
y_test = torch.tensor(data["y_test"], dtype=torch.float32).view(-1,1)

In [7]:
class LSTMModel(nn.Module):
    def __init__(self, input_size=10, hidden_size=128, num_layers=2):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.2
        )

        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)

        # take last timestep output
        out = out[:, -1, :]
        out = self.dropout(out)
        out = self.fc(out)
        return out

In [21]:
class BiLSTMModel(nn.Module):
    def __init__(self, input_size=10, hidden_size=128, num_layers=2):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.3,
            bidirectional=True
        )

        # because bidirectional → hidden_size * 2
        self.fc = nn.Linear(hidden_size * 2, 1)

    def forward(self, x):
        out, _ = self.lstm(x)

        # out = out[:, -1, :]   # last timestep
        out = out.mean(dim=1)
        out = self.fc(out)

        return out

In [16]:
class LSTMAttention(nn.Module):
    def __init__(self, input_size=10, hidden_size=128, num_layers=2):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.3
        )

        # attention layer
        self.attention = nn.Linear(hidden_size, 1)

        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)   # (batch, seq_len, hidden)

        # attention scores
        attn_weights = torch.softmax(self.attention(lstm_out), dim=1)

        # weighted sum
        context = torch.sum(attn_weights * lstm_out, dim=1)

        out = self.fc(context)
        return out

In [22]:
# model = LSTMModel()
model = BiLSTMModel()
#model = LSTMAttention()

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005)

In [23]:
epochs = 50
batch_size = 128

for epoch in range(epochs):

    perm = torch.randperm(X_train.size(0))
    total_loss = 0

    for i in range(0, X_train.size(0), batch_size):
        idx = perm[i:i+batch_size]

        xb = X_train[idx]
        yb = y_train[idx]

        pred = model(xb)
        loss = criterion(pred, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / (X_train.size(0) // batch_size)

    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")

Epoch 1, Loss: 5542.9177
Epoch 2, Loss: 3898.2975
Epoch 3, Loss: 2996.8163
Epoch 4, Loss: 2300.3467
Epoch 5, Loss: 1745.8124
Epoch 6, Loss: 1371.4106
Epoch 7, Loss: 1093.1607
Epoch 8, Loss: 840.3947
Epoch 9, Loss: 662.8629
Epoch 10, Loss: 528.5339
Epoch 11, Loss: 424.9579
Epoch 12, Loss: 358.2278
Epoch 13, Loss: 304.0408
Epoch 14, Loss: 254.5977
Epoch 15, Loss: 223.3729
Epoch 16, Loss: 206.7810
Epoch 17, Loss: 182.8619
Epoch 18, Loss: 166.8196
Epoch 19, Loss: 154.7631
Epoch 20, Loss: 157.5651
Epoch 21, Loss: 139.9540
Epoch 22, Loss: 124.0378
Epoch 23, Loss: 122.2702
Epoch 24, Loss: 116.7690
Epoch 25, Loss: 114.7621
Epoch 26, Loss: 104.0470
Epoch 27, Loss: 106.6145
Epoch 28, Loss: 93.4388
Epoch 29, Loss: 92.8132
Epoch 30, Loss: 89.9095
Epoch 31, Loss: 85.3041
Epoch 32, Loss: 83.2222
Epoch 33, Loss: 79.5308
Epoch 34, Loss: 74.9256
Epoch 35, Loss: 73.3486
Epoch 36, Loss: 72.1230
Epoch 37, Loss: 71.7564
Epoch 38, Loss: 76.1646
Epoch 39, Loss: 62.8688
Epoch 40, Loss: 63.9172
Epoch 41, Loss:

In [ ]:
model.eval()

with torch.no_grad():
    pred = model(X_test).numpy()

rmse = np.sqrt(mean_squared_error(y_test.numpy(), pred))

# print("LSTM RMSE:", rmse)
print("LSTM-attention RMSE:", rmse)


# Bi-LSTM RMSE: 13.495558396581702 (when using last timestep output)
# LSTM-attention RMSE: 12.449974360305552
# mean polling bi-lstm rmse: 13.44449820039229


LSTM-attention RMSE: 13.44449820039229
